In [4]:
import os

import numpy as np
import json
from prettytable import PrettyTable

opticflowModelList = ('RAFT', 'SEA_RAFT',
                      'MemFlow', 'StreamFlow', 'DpFlow', 'FlowDiffuser') 
directionalStmdList = ('DSTMD', 'STMDPlus', 'ApgSTMD', 'vSTMD', 'vSTMD_F') 
directionModelList = opticflowModelList + directionalStmdList
ingore_velocity = [6500, 7500, 8500, 9500]

def show_table():
    with open('direction_error_across_velocity.json', 'r') as f:
        data = json.load(f)
        vList = data['velocity']
        errList = data['errDict']


    errTable = PrettyTable()
    errTable.title = "Table. AEE across Different Velocities (pixels/frame)"
    errTable.field_names = ['Model'] + [f'{i*0.1:.1f}' for i in range(1, 10)] + [f'{i*0.2:.1f}' for i in range(5, 10)] + \
        [f'{i*0.5:.1f}' for i in range(4, 12)] + [i for i in range(6, 11)] + ['Avg.Range']
    
    AAE_table = PrettyTable()
    AAE_table.title = "Table II. AAE across Different Velocities (pixels/frame)"
    AAE_table.field_names = ['Model', '0.1 - 0.5', '0.6 - 1.0', '1.1 - 3', '3.1 - 5', '5.1 - 10', 'Avg.Range']

    rangeList = [0 for _ in directionModelList]
    for i, v in enumerate(vList):
        if v in ingore_velocity:
            continue
        AEEList = [errList[modelname][i] if modelname in errList.keys() else -1 for modelname in directionModelList]
        argsortedList = np.argsort(AEEList)
        for j, r in enumerate(argsortedList):
            rangeList[r] += j + 1

    argsortIdx = np.argsort(rangeList)
    argsortAvgRange = [None for _ in range(len(directionModelList))]
    
    for k in range(len(directionModelList)):
        argsortAvgRange[argsortIdx[k]] = k + 1


    for modelname in directionModelList:
        if modelname in errList.keys():
            _row = [modelname, ]
            AAE_0_1_to_0_5 = []
            AAE_0_6_to_1_0 = []
            AAE_1_1_to_3_0 = []
            AAE_3_1_to_5_0 = []
            AAE_5_1_to_10_0 = []
            for i, v in enumerate(vList):
                if v >= 100 and v <= 500:
                    AAE_0_1_to_0_5.append(errList[modelname][i])
                elif v > 500 and v <= 1000:
                    AAE_0_6_to_1_0.append(errList[modelname][i])
                elif v > 1000 and v <= 3000:
                    AAE_1_1_to_3_0.append(errList[modelname][i])
                elif v > 3000 and v <= 5000:
                    AAE_3_1_to_5_0.append(errList[modelname][i])
                elif v > 5000 and v <= 10000:
                    AAE_5_1_to_10_0.append(errList[modelname][i])
                if v in ingore_velocity:
                    continue
                _row.append(round(errList[modelname][i], 2))

            _row.append(argsortAvgRange[directionModelList.index(modelname)])
            errTable.add_row(_row)

            AAE_row = [modelname, 
                       round(np.mean(AAE_0_1_to_0_5), 2),
                       round(np.mean(AAE_0_6_to_1_0), 2),
                       round(np.mean(AAE_1_1_to_3_0), 2),
                       round(np.mean(AAE_3_1_to_5_0), 2),
                       round(np.mean(AAE_5_1_to_10_0), 2),
                       argsortAvgRange[directionModelList.index(modelname)]
                       ]
            AAE_table.add_row(AAE_row)


    # Print the tables
    print(errTable)
    print(AAE_table)


if __name__ == "__main__":
    show_table()

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                 Table. AEE across Different Velocities (pixels/frame)                                                                                 |
+--------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-----------+
|    Model     | 0.1  | 0.2  | 0.3  | 0.4  | 0.5  | 0.6  | 0.7  | 0.8  | 0.9  | 1.0  | 1.2  | 1.4  | 1.6  | 1.8  | 2.0  | 2.5  | 3.0  | 3.5  | 4.0  | 4.5  | 5.0  | 5.5  |  6   |  7   |  8   |  9   |  10  | Avg.Range |
+--------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+